# GPT-2 (124M) — Complete Architecture & Pipeline

> A self-contained reference that assembles **every building block** of GPT-2 in one notebook:  
> Multi-Head Attention → LayerNorm → GELU → FeedForward → TransformerBlock → GPTModel  
> → Text Generation → Training Loop → Pretrained Weight Loading

Based on *"Build a Large Language Model (From Scratch)"* by **Sebastian Raschka** (Manning, 2024).

In [ ]:
import torch
import torch.nn as nn
import tiktoken
import numpy as np
import os
import time

# ── GPT-2 124M Configuration ────────────────────────────────────
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size (BPE tokens)
    "context_length": 1024, # Maximum sequence length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of transformer blocks
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

# ── Tokenizer ───────────────────────────────────────────────────
tokenizer = tiktoken.get_encoding("gpt2")

# ── Device ──────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [ ]:
# ════════════════════════════════════════════════════════════════
# BLOCK 1 — Multi-Head Causal Attention
# ════════════════════════════════════════════════════════════════

class MultiHeadAttention(nn.Module):
    """Multi-head causal self-attention with scaled dot-product."""

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # dimension per head

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys    = self.W_key(x)    # (b, T, d_out)
        queries = self.W_query(x)
        values  = self.W_value(x)

        # Split into heads: (b, T, d_out) → (b, T, n_heads, head_dim) → (b, n_heads, T, head_dim)
        keys    = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values  = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention with causal mask
        attn_scores = queries @ keys.transpose(2, 3)                         # (b, h, T, T)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Combine heads: (b, h, T, head_dim) → (b, T, d_out)
        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)

In [ ]:
# ════════════════════════════════════════════════════════════════
# BLOCK 2 — Layer Normalization
# ════════════════════════════════════════════════════════════════

class LayerNorm(nn.Module):
    """Layer normalization with learnable scale & shift."""

    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var  = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


# ════════════════════════════════════════════════════════════════
# BLOCK 3 — GELU Activation
# ════════════════════════════════════════════════════════════════

class GELU(nn.Module):
    """Gaussian Error Linear Unit (tanh approximation)."""

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


# ════════════════════════════════════════════════════════════════
# BLOCK 4 — Feed-Forward Network
# ════════════════════════════════════════════════════════════════

class FeedForward(nn.Module):
    """Two-layer MLP with 4× expansion and GELU activation."""

    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),   # Expansion
            GELU(),                                           # Activation
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),   # Contraction
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
# ════════════════════════════════════════════════════════════════
# BLOCK 5 — Transformer Block (Pre-Norm with Residual Connections)
# ════════════════════════════════════════════════════════════════

class TransformerBlock(nn.Module):
    """Pre-norm Transformer block: LayerNorm → Attention → Residual → LayerNorm → FFN → Residual."""

    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Attention block with residual connection
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        # Feed-forward block with residual connection
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x

## 2 — Complete GPT Model

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/14.webp?1" width="400px">

Token Embeddings + Positional Embeddings → Dropout → N × TransformerBlock → LayerNorm → Linear Output Head

In [ ]:
# ════════════════════════════════════════════════════════════════
# BLOCK 6 — Complete GPT Model
# ════════════════════════════════════════════════════════════════

class GPTModel(nn.Module):
    """Full GPT-2 architecture: embeddings → transformer blocks → output logits."""

    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # (batch, seq_len, emb_dim)
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [ ]:
# ── Instantiate the model and verify shapes ─────────────────────
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

# Quick test with a small batch
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)

out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)  # Expected: (2, 4, 50257)

Input batch:
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

Output shape: torch.Size([2, 4, 50257])
tensor([[[ 0.3613,  0.4222, -0.0711,  ...,  0.3483,  0.4661, -0.2838],
         [-0.1792, -0.5660, -0.9485,  ...,  0.0477,  0.5181, -0.3168],
         [ 0.7120,  0.0332,  0.1085,  ...,  0.1018, -0.4327, -0.2553],
         [-1.0076,  0.3418, -0.1190,  ...,  0.7195,  0.4023,  0.0532]],

        [[-0.2564,  0.0900,  0.0335,  ...,  0.2659,  0.4454, -0.6806],
         [ 0.1230,  0.3653, -0.2074,  ...,  0.7705,  0.2710,  0.2246],
         [ 1.0558,  1.0318, -0.2800,  ...,  0.6936,  0.3205, -0.3178],
         [-0.1565,  0.3926,  0.3288,  ...,  1.2630, -0.1858,  0.0388]]],
       grad_fn=<UnsafeViewBackward0>)


In [ ]:
# ── Parameter Counting ──────────────────────────────────────────
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

# With weight tying (tok_emb and out_head share weights in GPT-2)
total_params_gpt2 = total_params - sum(p.numel() for p in model.out_head.parameters())
print(f"Parameters with weight tying: {total_params_gpt2:,}")

# Memory footprint (FP32)
total_size_mb = total_params * 4 / (1024 * 1024)
print(f"Model size: {total_size_mb:.2f} MB")

print(f"\nToken embedding shape: {model.tok_emb.weight.shape}")
print(f"Output head shape:     {model.out_head.weight.shape}")

## 3 — Text Generation

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/15.webp" width="400px">

In [ ]:
# ── Utility: text ↔ token IDs ───────────────────────────────────

def text_to_token_ids(text, tokenizer):
    """Encode text to a batched tensor of token IDs."""
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    return torch.tensor(encoded).unsqueeze(0)  # (1, seq_len)

def token_ids_to_text(token_ids, tokenizer):
    """Decode a batched tensor of token IDs back to text."""
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())

In [ ]:
# ── Greedy Decoding (argmax) ────────────────────────────────────

def generate_text_simple(model, idx, max_new_tokens, context_size):
    """Generate tokens one-by-one using greedy argmax decoding."""
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]          # crop to context window
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]                  # last token logits
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

In [ ]:
# ── Advanced Generation (Temperature + Top-k + EOS stopping) ────

def generate(model, idx, max_new_tokens, context_size,
             temperature=0.0, top_k=None, eos_id=None):
    """Generate tokens with temperature scaling and top-k sampling."""
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        # Top-k filtering
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(
                logits < min_val,
                torch.tensor(float("-inf")).to(logits.device),
                logits
            )

        # Temperature scaling + sampling
        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        if idx_next == eos_id:
            break

        idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [ ]:
# ── Test: Generate with untrained model (random weights) ────────
model.eval()
model.to(device)

start_context = "Hello, I am"
encoded = text_to_token_ids(start_context, tokenizer).to(device)

out = generate_text_simple(
    model=model,
    idx=encoded,
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Prompt:", start_context)
print("Generated:", token_ids_to_text(out, tokenizer))
print("(Random output expected — model is untrained)")

## 4 — Training on "The Verdict"

Train the model from scratch on a small corpus to verify the training loop works end-to-end.

In [ ]:
# ── Dataset & DataLoader ────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader
import urllib.request

class GPTDatasetV1(Dataset):
    """Sliding-window dataset that creates input/target token pairs."""

    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk  = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        drop_last=drop_last, num_workers=num_workers
    )

In [ ]:
# ── Load training corpus ────────────────────────────────────────
file_path = "../Data/the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    with urllib.request.urlopen(url) as response:
        text_data = response.read().decode("utf-8")
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as f:
        text_data = f.read()

print(f"Characters: {len(text_data):,}")
print(f"Tokens:     {len(tokenizer.encode(text_data)):,}")
print(f"First 100 chars: {text_data[:100]!r}")

In [ ]:
# ── Use shorter context length for training on small data ───────
TRAIN_CONFIG = GPT_CONFIG_124M.copy()
TRAIN_CONFIG["context_length"] = 256  # smaller context for efficiency

# 90/10 train/val split
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data   = text_data[split_idx:]

torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data, batch_size=2,
    max_length=TRAIN_CONFIG["context_length"],
    stride=TRAIN_CONFIG["context_length"],
    drop_last=True, shuffle=True
)

val_loader = create_dataloader_v1(
    val_data, batch_size=2,
    max_length=TRAIN_CONFIG["context_length"],
    stride=TRAIN_CONFIG["context_length"],
    drop_last=False, shuffle=False
)

print("Train loader:")
for x, y in train_loader:
    print(f"  {x.shape} → {y.shape}")

print("Validation loader:")
for x, y in val_loader:
    print(f"  {x.shape} → {y.shape}")

In [ ]:
# ── Loss Functions ──────────────────────────────────────────────

def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch  = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    return torch.nn.functional.cross_entropy(
        logits.flatten(0, 1), target_batch.flatten()
    )


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.0
    if len(data_loader) == 0:
        return float("nan")
    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i >= num_batches:
            break
        total_loss += calc_loss_batch(input_batch, target_batch, model, device).item()
    return total_loss / num_batches


def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss   = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))
    model.train()

In [ ]:
# ── Training Loop ───────────────────────────────────────────────

def train_model_simple(model, train_loader, val_loader, optimizer, device,
                       num_epochs, eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        generate_and_print_sample(model, tokenizer, device, start_context)

    return train_losses, val_losses, track_tokens_seen

In [ ]:
# ── Train! ──────────────────────────────────────────────────────
start_time = time.time()

torch.manual_seed(123)
trained_model = GPTModel(TRAIN_CONFIG).to(device)
optimizer = torch.optim.AdamW(trained_model.parameters(), lr=0.0004, weight_decay=0.1)

num_epochs = 10
train_losses, val_losses, tokens_seen = train_model_simple(
    trained_model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer
)

elapsed = (time.time() - start_time) / 60
print(f"\nTraining completed in {elapsed:.2f} minutes.")

In [ ]:
# ── Plot Training & Validation Loss ─────────────────────────────
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(6, 3.5))
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax2 = ax1.twiny()
    ax2.plot(tokens_seen, train_losses, alpha=0)
    ax2.set_xlabel("Tokens seen")
    fig.tight_layout()
    plt.show()

epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

In [ ]:
# ── Generate from trained model ─────────────────────────────────
trained_model.eval()

torch.manual_seed(123)
token_ids = generate(
    model=trained_model,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=25,
    context_size=TRAIN_CONFIG["context_length"],
    top_k=25,
    temperature=1.4
)
print("Generated:", token_ids_to_text(token_ids, tokenizer))

## 5 — Save & Load Model Weights

In [ ]:
# ── Save model + optimizer state ────────────────────────────────
save_path = "../Weights/model_and_optimizer.pth"
torch.save({
    "model_state_dict": trained_model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
}, save_path)
print(f"Saved checkpoint to {save_path}")

# ── Load model + optimizer state ────────────────────────────────
checkpoint = torch.load(save_path, weights_only=True)

loaded_model = GPTModel(TRAIN_CONFIG).to(device)
loaded_model.load_state_dict(checkpoint["model_state_dict"])
loaded_model.eval()

loaded_optimizer = torch.optim.AdamW(loaded_model.parameters(), lr=0.0004, weight_decay=0.1)
loaded_optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

print("Model and optimizer loaded successfully!")

## 6 — Load Pretrained OpenAI GPT-2 Weights

Download the official GPT-2 124M checkpoint from OpenAI, convert the TensorFlow weights to PyTorch, and run real text generation.

In [ ]:
# ── Download GPT-2 weights from OpenAI ──────────────────────────
import json
import requests
import tensorflow as tf
from tqdm import tqdm


def download_file(url, destination):
    """Download a file with progress bar, skipping if already present."""
    try:
        response = requests.get(url, stream=True, verify=False)
        file_size = int(response.headers.get("content-length", 0))

        if os.path.exists(destination):
            if os.path.getsize(destination) == file_size:
                print(f"  Already exists: {os.path.basename(destination)}")
                return

        block_size = 1024
        with tqdm(total=file_size, unit="iB", unit_scale=True,
                  desc=os.path.basename(destination)) as pbar:
            with open(destination, "wb") as f:
                for chunk in response.iter_content(block_size):
                    pbar.update(len(chunk))
                    f.write(chunk)
    except requests.exceptions.RequestException as e:
        print(f"Error: {e}")


def load_gpt2_params_from_tf_ckpt(ckpt_path, settings):
    """Parse TF checkpoint variables into a nested dict."""
    params = {"blocks": [{} for _ in range(settings["n_layer"])]}
    for name, _ in tf.train.list_variables(ckpt_path):
        variable_array = np.squeeze(tf.train.load_variable(ckpt_path, name))
        variable_name_parts = name.split("/")[1:]  # skip 'model/' prefix
        target_dict = params
        if variable_name_parts[0].startswith("h"):
            layer_number = int(variable_name_parts[0][1:])
            target_dict = params["blocks"][layer_number]
        for key in variable_name_parts[1:-1]:
            target_dict = target_dict.setdefault(key, {})
        target_dict[variable_name_parts[-1]] = variable_array
    return params


def download_and_load_gpt2(model_size, models_dir):
    """Download and load GPT-2 pretrained weights."""
    allowed = ("124M", "355M", "774M", "1558M")
    if model_size not in allowed:
        raise ValueError(f"Model size must be one of {allowed}")

    model_dir = os.path.join(models_dir, "gpt2", model_size)
    base_url = "https://openaipublic.blob.core.windows.net/gpt-2/models"
    filenames = [
        "checkpoint", "encoder.json", "hparams.json",
        "model.ckpt.data-00000-of-00001", "model.ckpt.index",
        "model.ckpt.meta", "vocab.bpe"
    ]

    os.makedirs(model_dir, exist_ok=True)
    for fn in filenames:
        file_url = f"{base_url}/{model_size}/{fn}"
        file_path = os.path.join(model_dir, fn)
        if not os.path.exists(file_path):
            print(f"Downloading {fn}...")
            download_file(file_url, file_path)
        else:
            print(f"  {fn} already exists.")

    tf_ckpt_path = tf.train.latest_checkpoint(model_dir)
    settings = json.load(open(os.path.join(model_dir, "hparams.json")))
    params = load_gpt2_params_from_tf_ckpt(tf_ckpt_path, settings)
    return settings, params


# Download GPT-2 124M
settings, params = download_and_load_gpt2(model_size="124M", models_dir="../Weights")
print("\nModel settings:", settings)

In [ ]:
# ── Map OpenAI weights into our GPTModel ────────────────────────

def assign(left, right):
    """Assign numpy weights to a PyTorch parameter, checking shapes."""
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))


def load_weights_into_gpt(gpt, params):
    """Load OpenAI GPT-2 weights into our GPTModel architecture."""
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params["wpe"])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params["wte"])

    for b in range(len(params["blocks"])):
        # Attention: split combined QKV weights
        q_w, k_w, v_w = np.split(
            params["blocks"][b]["attn"]["c_attn"]["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            params["blocks"][b]["attn"]["c_attn"]["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)

        # Attention output projection
        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"])

        # Feed-forward MLP
        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        # Layer norms
        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale, params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift, params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale, params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift, params["blocks"][b]["ln_2"]["b"])

    # Final layer norm & output head (weight tied with token embeddings)
    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight  = assign(gpt.out_head.weight, params["wte"])

In [ ]:
# ── Instantiate model with pretrained config and load weights ───
model_configs = {
    "gpt2-small (124M)":  {"emb_dim": 768,  "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)":  {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)":    {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

model_name = "gpt2-small (124M)"
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

gpt_pretrained = GPTModel(NEW_CONFIG)
load_weights_into_gpt(gpt_pretrained, params)
gpt_pretrained.to(device)
gpt_pretrained.eval()

print(f"Loaded pretrained {model_name} weights successfully!")

In [ ]:
# ── Generate text with pretrained GPT-2! ────────────────────────
prompts = [
    "Every effort moves you",
    "The meaning of life is",
    "In a shocking finding, scientists discovered",
]

for prompt in prompts:
    torch.manual_seed(123)
    token_ids = generate(
        model=gpt_pretrained,
        idx=text_to_token_ids(prompt, tokenizer).to(device),
        max_new_tokens=30,
        context_size=NEW_CONFIG["context_length"],
        top_k=50,
        temperature=1.2
    )
    print(f"\n{'─'*60}")
    print(f"Prompt:    {prompt}")
    print(f"Generated: {token_ids_to_text(token_ids, tokenizer)}")

## Summary

This notebook contains the **complete GPT-2 (124M) pipeline** in a single file:

| Section | What it covers |
|---------|---------------|
| **1 — Config & Imports** | Model hyperparameters, tokenizer, device setup |
| **2 — Architecture** | `MultiHeadAttention` → `LayerNorm` → `GELU` → `FeedForward` → `TransformerBlock` → `GPTModel` |
| **3 — Text Generation** | `generate_text_simple()` (greedy) and `generate()` (temperature + top-k + EOS) |
| **4 — Training** | `GPTDatasetV1`, `create_dataloader_v1()`, loss functions, `train_model_simple()`, loss plotting |
| **5 — Save & Load** | Checkpoint saving/loading with optimizer state |
| **6 — Pretrained Weights** | Download OpenAI GPT-2 → convert TF→PyTorch → `load_weights_into_gpt()` → real generation |

---

*Based on "Build a Large Language Model (From Scratch)" by Sebastian Raschka (Manning, 2024)*